In [ ]:
# Kaggle Compatibility & Path Config
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Auto-detect environment
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # Identify dataset directory on Kaggle
    DATA_DIR = '/kaggle/input/'
    # Standard Kaggle setup often puts CSVs in a subfolder. 
    # We search for the first folder inside /kaggle/input
    for d in os.listdir(DATA_DIR):
        if os.path.isdir(os.path.join(DATA_DIR, d)):
            DATA_DIR = os.path.join(DATA_DIR, d) + '/'
            break
    OUT_DIR = '/kaggle/working/'
else:
    DATA_DIR = '../data/raw/'
    OUT_DIR = '../output/'

TRAIN_FILE = os.path.join(DATA_DIR, 'sales.csv')
TEST_FILE  = os.path.join(DATA_DIR, 'sample_submission.csv')
OUT_FILE   = os.path.join(OUT_DIR, 'submission.csv')

print(f"Running on {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data Directory: {DATA_DIR}")

# Sales Forecasting — Baseline
**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01 using historical data (2012–2022).

**Strategy:** Seasonal profile + YoY growth trend.

## 2 — Load & Inspect Data

In [ ]:
train = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
test  = pd.read_csv(TEST_FILE,  parse_dates=['Date'])

print('Train shape:', train.shape)
print('Train date range:', train['Date'].min().date(), '→', train['Date'].max().date())
print()
print('Test shape:', test.shape)
print('Test date range:', test['Date'].min().date(), '→', test['Date'].max().date())
print()
print(train.tail())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(train['Date'], train['Revenue'], lw=0.7)
axes[0].set_title('Historical Daily Revenue'); axes[0].set_ylabel('Revenue')
axes[1].plot(train['Date'], train['COGS'], lw=0.7, color='orange')
axes[1].set_title('Historical Daily COGS'); axes[1].set_ylabel('COGS')
plt.tight_layout()
plt.show()

## 3 — Feature Engineering

In [ ]:
train['year']       = train['Date'].dt.year
train['day_of_year'] = train['Date'].dt.dayofyear
train['month']      = train['Date'].dt.month
train['day']        = train['Date'].dt.day

# Annual totals — used to estimate YoY growth
annual = train.groupby('year')[['Revenue', 'COGS']].sum()
print('Annual totals (only complete years shown):')
print(annual)

In [ ]:
# --- YoY growth rate (geometric mean, 2013–2022) ---
full_years = annual.loc[2013:2022]
yoy_rev  = full_years['Revenue'].pct_change().dropna()
yoy_cogs = full_years['COGS'].pct_change().dropna()
growth_rev  = (1 + yoy_rev).prod() ** (1 / len(yoy_rev))
growth_cogs = (1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))

print(f'Geometric mean YoY Revenue growth : {growth_rev:.4f}  ({(growth_rev-1)*100:.2f}%/yr)')
print(f'Geometric mean YoY COGS    growth : {growth_cogs:.4f}  ({(growth_cogs-1)*100:.2f}%/yr)')

## 4 — Build Seasonal Profile

In [ ]:
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')
train['rev_norm']  = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS']    / annual_means['COGS']

seasonal = (
    train
    .groupby(['month', 'day'])[['rev_norm', 'cogs_norm']]
    .mean()
    .reset_index()
)
print('Seasonal profile rows:', len(seasonal))

## 5 — Predict Test Period

In [ ]:
base_rev  = annual.loc[2022, 'Revenue']  / 365
base_cogs = annual.loc[2022, 'COGS']     / 365

test = test.copy()
test['month'] = test['Date'].dt.month
test['day']   = test['Date'].dt.day
test['year']  = test['Date'].dt.year
test['years_ahead'] = test['year'] - 2022

test = test.merge(seasonal, on=['month', 'day'], how='left')
test['rev_norm']  = test['rev_norm'].fillna(1.0)
test['cogs_norm'] = test['cogs_norm'].fillna(1.0)

test['Revenue_pred'] = (base_rev  * growth_rev**test['years_ahead']  * test['rev_norm'] ).round(2)
test['COGS_pred']    = (base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']).round(2)

print('Predictions sample:')
print(test[['Date','Revenue_pred','COGS_pred']].head(10))

## 6 — Export Submission

In [ ]:
submission = test[['Date', 'Revenue_pred', 'COGS_pred']].rename(
    columns={'Revenue_pred': 'Revenue', 'COGS_pred': 'COGS'}
)
submission['Date'] = pd.to_datetime(submission['Date']).dt.strftime('%Y-%m-%d')
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
submission.to_csv(OUT_FILE, index=False)

print(f'Saved {len(submission)} rows to {OUT_FILE}')